# ARCHELEC CORPUS PREPROCESSING - LEGISLATIVE ELECTIONS (1973-1993) 

This notebook implements preprocessing of the Archelec corpus (directly available in https://gitlab.teklia.com/ckermorvant/arkindex_archelec/-/tree/master/text_files?ref_type=heads). It is restricted to legislative elections spanning from 1973 to 1993. 

The text preprocessing encompasses :
- Text normalization : it keeps exclusively alphabetic characters, including those with accents, and remove formatting issues (i.e., collapsing multiple spaces into a single one).
- Word-tokenization : lemmatization, general French stopwords removal, and keep exclusively nouns, adjectives and proper nouns using POS filtering. Bigrams are subsequently obtained and custom stopwords manually identified removed. 
- Rare tokens removal. 

# Set Up the Environment

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from gensim.models import Phrases
from gensim.models.phrases import Phraser
import re
import unicodedata


# Define paths relative to this notebook's location (script/ folder)
current_dir = Path.cwd()
project_root = current_dir.parent
meta_dir = project_root / "metadata"
data_dir = project_root / "data"
spacy_model_path = project_root / "fr_core_news_md-3.8.0-py3-none-any.whl"
nlp = spacy.load("fr_core_news_md")

The code below reads the metadata file and merges the texts to each candidate. 

In [4]:
meta_file = meta_dir / "archelect_search.csv" 

print(f"Checking: {meta_file}")

if meta_file.exists():
    print("Found metadata file!")
    master_df = pd.read_csv(meta_file)
    
    # 1. Parse dates and extract the year
    master_df['parsed_date'] = pd.to_datetime(master_df['date'], errors='coerce')
    
    # 2. Filter out rows with unparseable dates, then format the year as a string for the file path
    master_df = master_df.dropna(subset=['parsed_date']).copy()
    master_df['year'] = master_df['parsed_date'].dt.year.astype(int).astype(str)
    
    # 3. Define the helper function to fetch text
    def fetch_text(row, data_dir):
        manifesto_id = row['id']
        year = row['year']
        
        # Matches data/ID.txt
        txt_path = data_dir / f"{manifesto_id}.txt"
        
        if txt_path.exists():
            try:
                return txt_path.read_text(encoding='utf-8')
            except Exception as e:
                print(f"Error reading {txt_path}: {e}")
                return None
        return None

    # 4. Apply the function across rows (axis=1)
    print("Fetching text files... This might take a moment.")
    master_df['text'] = master_df.apply(lambda row: fetch_text(row, data_dir), axis=1)
    
    # Clean-up: remove the temporary parsed_date column
    master_df = master_df.drop(columns=['parsed_date'])
    
    print(f"Success! Processed DataFrame with {len(master_df)} rows.")
    
    # Print a quick summary of how many texts were successfully found
    missing_texts = master_df['text'].isna().sum()
    print(f"Found texts for {len(master_df) - missing_texts} out of {len(master_df)} records.")

else:
    print(f"Metadata file not found at: {meta_file.absolute()}")

Checking: c:\Users\mmlin\Projects\NLP\Project\metadata\archelect_search.csv
Found metadata file!


C:\Users\mmlin\AppData\Local\Temp\ipykernel_47332\3301384666.py:7: DtypeWarning: Columns (0: departement-nom, 1: departement-insee, 2: identifiant de circonscription, 3: pdf, 4: suppleant-nom, 5: suppleant-prenom, 6: suppleant-sexe, 7: suppleant-age, 8: suppleant-age-calcule, 9: suppleant-age-tranche, 10: suppleant-profession, 11: suppleant-mandat-en-cours, 12: suppleant-mandat-passe, 13: suppleant-associations, 14: suppleant-autres-statuts, 15: suppleant-soutien, 16: suppleant-liste, 17: suppleant-decorations) have mixed types. Specify dtype option on import or set low_memory=False.
  master_df = pd.read_csv(meta_file)


Fetching text files... This might take a moment.
Success! Processed DataFrame with 33031 rows.
Found texts for 21167 out of 33031 records.


We restrict our attention to legislative elections in France from 1973 to 1993. 

In [5]:
# Filter to the specific years of interest
years = ["1973","1978", "1981", "1988", "1993"]
master_df=master_df[master_df['year'].isin(years)]
# Check the distribution of texts across the selected years
year_counts = master_df['year'].value_counts().sort_index()
print("Text counts by year:")
print(year_counts)
# Total number of texts
total_texts = len(master_df)
print(f"Total number of texts in the filtered dataset: {total_texts}")
# Remove rows where 'text' is NaN (i.e., text file was not found or could not be read)
master_df = master_df.dropna(subset=['text'])
print(f"Number of texts after dropping missing ones: {len(master_df)}")

Text counts by year:
year
1973    3843
1978    4830
1981    3133
1988    3551
1993    5837
Name: count, dtype: int64
Total number of texts in the filtered dataset: 21194
Number of texts after dropping missing ones: 21167


# TEXT CLEANING

## Basic normalization

In [ ]:
def spacy_compatible_clean(text):
    """
    Cleaning that preserves accents, apostrophes, and casing.
    """
    if not isinstance(text, str):
        return ""
        
    # 1. Remove numbers 
    cleaned_text = re.sub(r'\d+', '', text)
    
    # 2. Collapse multiple spaces, tabs, or newlines into a single space
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

# Apply the lightweight cleaning function
master_df['cleaned_text'] = master_df['text'].apply(spacy_compatible_clean)


# Tokenization

## STOPWORD definition

In [ ]:
# 1. Load general grammatical French stopwords
GENERAL_STOPWORDS = set([x.strip().lower() for x in open(data_dir / 'stop_word_fr.txt').readlines()])

# 2. Define domain-specific stopwords
raw_custom_stopwords = {
    "monsieur", "madame","messieurs","mesdames", "mademoiselle", "mesdemoiselles","candidat", "candidate", "candidats","gauche","droite", "élection", 
    "vote", "voter", "circonscription", "département", "république", "sciences", "po", "cevipof", "française", "fonds",
    "liberté", "égalité", "fraternité",  "électeur", "électrice", "électeurs", "électrices", "parti", "partis",
    "janvier", "février", "mars", "avril", "mai", "juin", "juillet", "août", "septembre", "octobre", "novembre", "décembre",
    "politique", "france", "français", "française", "françaises", "français", "national", "nationale", "nationaux",
    "élection", "élections", "électoral", "électorale", "électorales", "électoraux", "législative", "législatives", "présidentielle", "présidentielles", "municipale", "municipales",
    "majorité", "minorité", "gouvernement", "opposition", "parlement", "assemblée", "sénat", "député", "députée", "députés", "députées",
    "suppléant", "suppléante", "suppléants", "suppléantes", "programme" , "programmes","commun", "francaise", "francais", 
    "francaises", "action", "moyen", "mesure", "mesures", "projets", "projet", "avenir", "changement", "ensemble", "pays", "citoyen", "citoyenne", 
    "Ain","Aisne","Allier","Alpes-de-Haute-Provence","Hautes-Alpes","Alpes-Maritimes","Ardèche","Ardennes","Ariège","Aube","Aude","Aveyron","Bouches-du-Rhône",
    "Calvados","Cantal","Charente","Charente-Maritime","Cher","Corrèze","Côte-d'Or","Côtes-d'Armor","Creuse","Dordogne","Doubs","Drôme","Eure","Eure-et-Loir",
    "Finistère","Corse-du-Sud","Haute-Corse","Gard","Haute-Garonne","Gers","Gironde","Hérault","Ille-et-Vilaine","Indre","Indre-et-Loire","Isère","Jura","Landes",
    "Loir-et-Cher","Loire","Haute-Loire","Loire-Atlantique","Loiret","Lot","Lot-et-Garonne","Lozère","Maine-et-Loire","Manche","Marne","Haute-Marne","Mayenne",
    "Meurthe-et-Moselle","Meuse","Morbihan","Moselle","Nièvre","Nord","Oise","Orne","Pas-de-Calais","Puy-de-Dôme","Pyrénées-Atlantiques","Hautes-Pyrénées",
    "Pyrénées-Orientales","Bas-Rhin","Haut-Rhin","Rhône","Haute-Saône","Saône-et-Loire","Sarthe","Savoie","Haute-Savoie","Paris","Seine-Maritime","Seine-et-Marne",
    "Yvelines","Deux-Sèvres","Somme","Tarn","Tarn-et-Garonne","Var","Vaucluse","Vendée","Vienne","Haute-Vienne","Vosges","Yonne","Territoire de Belfort","Essonne",
    "Hauts-de-Seine","Seine-Saint-Denis","Val-de-Marne","Val-d'Oise","Guadeloupe","Martinique","Guyane","La Réunion","Mayotte", "mitterrand", "françois", "jean", "lepen", "marine", 
    "social", "sociaux", "sociale", "vie", "pouvoir", "pouvoirs", "peuple"}


# Clean and lemmatize the custom stopwords
custom_stopwords_lemmatized = set()

for doc in nlp.pipe(raw_custom_stopwords, disable=['parser', 'ner']):
    for token in doc:
        custom_stopwords_lemmatized.add(token.lemma_.lower())

print(f"Loaded {len(GENERAL_STOPWORDS)} general stopwords.")
print(f"Loaded {len(custom_stopwords_lemmatized)} custom lemmatized stopwords.")

# Define non-lemmatized stopwords for direct addition to the set after lemmatization
raw_stopwords = {"departement", "circonscription", "republique", "republiqu", "liberte", "egalite", "fraternite", 
    "ère", "ere", "ème", "eme", "election", "legislative", "legislatives", "egalité", "sciences", "po", "cevipofscience", "fonds", "président", "république", "michel"
    , "rocard", "bulletin", "degré", "jean", "marie", "pen", "côté", "dimanche", "prochain", "prochaine", "arlette", "laguiller", "grand", "rpr", "udf", "votez", "suppléant",
    "sppléants", "suppléante", "suppléantes"} # I chose these manually based on their frequency in the tokens after 

# Add the additional stopwords to the lemmatized set
custom_stopwords_lemmatized.update(raw_stopwords)


Loaded 701 general stopwords.
Loaded 186 custom lemmatized stopwords.


## Remove general STOPWORDS, lemmatize and POS filter 

In [ ]:
ALLOWED_POS = {'NOUN', 'ADJ', 'PROPN'}

def extract_lemmas_base(texts):
    """
    Extracts lemmas, filters POS, and removes standard grammatical stopwords.
    """
    lemmatized_list = []
    
    for doc in nlp.pipe(texts, disable=['parser', 'ner']):
        lemmas = [
            token.lemma_.lower() for token in doc 
            if token.is_alpha 
            and not token.is_space                             
            and token.pos_ in ALLOWED_POS                     
            and token.text.lower() not in GENERAL_STOPWORDS           
            and token.lemma_.lower() not in GENERAL_STOPWORDS         
        ]
        lemmatized_list.append(lemmas)
        
    return lemmatized_list

master_df['lemmas_base'] = extract_lemmas_base(master_df['cleaned_text'].astype(str))


# Clean non-alphabetic symbols

In [ ]:
VALID_WORD_PATTERN = re.compile(r'^[a-zA-ZàâäéèêëïîôöùûüÿçÀÂÄÉÈÊËÏÎÔÖÙÛÜŸÇ\-]+$')

def remove_symbols_from_tokens(token_list):
    """
    Filters a list of tokens, keeping only pure alphabetical words and hyphenated words.
    """
    if not isinstance(token_list, list):
        return []
        
    cleaned_tokens = []
    for token in token_list:
        # Check if the token perfectly matches our valid characters rule
        if VALID_WORD_PATTERN.match(token) and token != '-':
            cleaned_tokens.append(token)
            
    return cleaned_tokens

# Apply the symbol filter to base lemmas
master_df['lemmas_base_clean'] = master_df['lemmas_base'].apply(remove_symbols_from_tokens)


# N-grams

In [ ]:
# Function to apply n-gram generation
def apply_ngrams(tokenized_docs, min_count=5, threshold=10):
    """
    Statistical N-gram Generation using Gensim.
    Stitches frequently co-occurring words together with an underscore.
    """
    
    bigram_model = Phrases(tokenized_docs, min_count=min_count, threshold=threshold)
    
    bigram_phraser = Phraser(bigram_model)
    
    # Apply the bigram model to the documents
    docs_with_bigrams = [bigram_phraser[doc] for doc in tokenized_docs]
    
    return docs_with_bigrams

master_df['lemmas_with_ngrams'] = apply_ngrams(master_df['lemmas_base_clean'])


# Custom STOPWORDS removal

In [ ]:
def smart_domain_filter(tokenized_docs, custom_stops):
    """
    Filters custom stopwords and bi-grams if both tokens are in custom stopwords.
    """
    
    stop_set = {(word) for word in custom_stops}
    
    cleaned_docs = []
    
    for doc in tokenized_docs:
        doc_clean = []
        for token in doc:
            
            
            if '_' in token:
                parts = token.split('_')
                
                if all(part in stop_set for part in parts):
                    continue 
                else:
                    doc_clean.append(token)
            
            else:
                if token not in stop_set:
                    doc_clean.append(token)
                    
        cleaned_docs.append(doc_clean)
        
    return cleaned_docs

# Apply the filter
master_df['final_tokens'] = smart_domain_filter(
    master_df['lemmas_with_ngrams'], 
    custom_stopwords_lemmatized
)
print(master_df[['cleaned_text', 'final_tokens']].head())


                                            cleaned_text  \
10776  Sciences Po / fonds CEVIPOF REPUBLIQUE FRANÇAI...   
10777  REPUBLIQUE FRANCAISE - LIBERTE - EGALITE - FRA...   
10778  Sciences Po / fonds CEVIPOF REPUBLIQUE FRANÇAI...   
10779  Sciences Po / fonds CEVIPOF REPUBLIQUE FRANÇAI...   
10780  Sciences Po / fonds CEVIPOF REPUBLIQUE FRANÇAI...   

                                            final_tokens  
10776  [paul_barberot, bourg-en-bresse, centr, progre...  
10777  [bourg_bresse, union, socialiste_democrate, pa...  
10778  [confiance, jeune, chomarat, ecole, livre, lyo...  
10779  [fraternité_departement, marcel_benoit, cultiv...  
10780  [confiance, lien, sang, amitié, suffrage, étiq...  


# LEAST COMMON TOKENS REMOVAL

In [ ]:
# Check list of 100 most frequent tokens and least frequent tokens 
from collections import Counter 
all_tokens = [token for doc in master_df['final_tokens'] for token in doc]
token_counts = Counter(all_tokens)
print("Most common tokens after n-gram generation:")
print(token_counts.most_common(100))
print("Least common tokens after n-gram generation:")
print(token_counts.most_common()[-100:])

Most common tokens after n-gram generation:
[('travailleur', 26321), ('emploi', 19657), ('union', 18311), ('homme', 17794), ('socialiste', 16314), ('travail', 15436), ('communiste', 15402), ('droit', 15319), ('entreprise', 14886), ('confiance', 13038), ('société', 12882), ('femme', 11989), ('maire', 11877), ('jeune', 11558), ('progrès', 11401), ('région', 11077), ('économique', 10731), ('tour', 10621), ('voix', 9868), ('pourcent', 9664), ('parti_communiste', 9454), ('parti_socialiste', 9445), ('chômage', 9350), ('enfant', 8476), ('service', 8377), ('choix', 8108), ('famille', 7997), ('développement', 7979), ('salaire', 7761), ('lutte', 7619), ('besoin', 7493), ('problème', 7433), ('intérêt', 7409), ('année', 7025), ('public', 6906), ('conseiller_général', 6875), ('volonté', 6800), ('retraite', 6730), ('économie', 6622), ('solidarité', 6590), ('suffrage', 6586), ('défense', 6444), ('europe', 6398), ('temps', 6353), ('place', 6261), ('monde', 6186), ('etat', 6142), ('véritable', 6117), (

In [10]:
# Remove tokens appearning in less than 5 documents
token_counts = Counter(token for doc in master_df['final_tokens'] for token in set(doc))
tokens_to_keep = {token for token, count in token_counts.items() if count >= 5}
master_df['final_tokens'] = master_df['final_tokens'].apply(lambda doc: [token for token in doc if token in tokens_to_keep])

# SAVE DATASET

In [ ]:
# Save the dataframe to a parquet file 
#!pip install pyarrow

# 1. Convert lists into space-separated strings 
master_df['final_tokens'] = master_df['final_tokens'].apply(
    lambda x: ' '.join(map(str, x)) if isinstance(x, list) else str(x)
)

# 2. Ensure all text columns are standard strings 
df_save=master_df.copy()
df_save['identifiant de circonscription'] = df_save['identifiant de circonscription'].astype(str)

# 3. Save to parquet
file_path = data_dir / 'cleaned_master_data.parquet'
df_save.to_parquet(file_path, engine='pyarrow')
